# ProofWriter + CLUTRR Neuro-Symbolic Reasoning Datasets

## Overview
This notebook demonstrates two peer-reviewed neuro-symbolic reasoning benchmark datasets:
- **ProofWriter** (tasksource/proofwriter, ACL 2021): 230K+ examples with gold proof traces
- **CLUTRR** (kendrivp/CLUTRR_v1_extracted, EMNLP 2019): 95K+ examples with kinship reasoning

Both datasets are standardized to a unified `exp_sel_data_out` schema for evaluating trace-graph auditing in neuro-symbolic pipelines.

## 1. Install Dependencies
Install required packages, checking if running on Colab to avoid reinstalling pre-installed packages.

In [ ]:
import subprocess, sys

def _pip(*packages):
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
    except Exception as e:
        print(f"Warning: pip install failed ({e}). Packages may already be installed or unavailable.")

try:
    _pip('loguru==0.7.3')
except:
    pass

if 'google.colab' not in sys.modules:
    try:
        _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')
    except:
        pass

## 2. Imports
Import all required libraries for data loading and analysis.

In [ ]:
import json
import re
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from loguru import logger
import sys

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

## 3. Data Loading Helper
Load mini_demo_data.json from GitHub (with local fallback for offline use).

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-0b8201-typed-failure-recovery-in-neuro-symbolic/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        logger.debug(f"GitHub load failed: {e}, trying local file")
        pass
    if Path("mini_demo_data.json").exists():
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local path")

## 4. Load Data
Load the demo dataset and display basic info.

In [ ]:
data = load_data()
logger.info(f"Loaded {len(data['datasets'])} datasets")
for ds in data['datasets']:
    logger.info(f"  {ds['dataset']}: {len(ds['examples'])} examples")

## 5. Configuration
Set tunable parameters to minimal values for demo. These can be scaled up for larger runs.

In [ ]:
MAX_EXAMPLES_PER_DATASET = 3  # Demo: show first 3 from each dataset
SHOW_FULL_INPUT = False  # If True, show full input text; if False, truncate at 100 chars
SHOW_FULL_PROOF = False  # If True, show full proof_state/allProofs; if False, truncate at 200 chars

logger.info(f"Config: MAX_EXAMPLES={MAX_EXAMPLES_PER_DATASET}, SHOW_FULL_INPUT={SHOW_FULL_INPUT}, SHOW_FULL_PROOF={SHOW_FULL_PROOF}")

## 6. Extract and Summarize Examples
Parse examples from each dataset, applying config limits.

In [ ]:
all_examples = []

for ds_info in data['datasets']:
    dataset_name = ds_info['dataset']
    examples = ds_info['examples'][:MAX_EXAMPLES_PER_DATASET]
    logger.info(f"Processing {dataset_name}: {len(examples)} examples")
    
    for i, ex in enumerate(examples):
        record = {
            'dataset': dataset_name,
            'example_index': i,
            'input': ex.get('input', ''),
            'output': ex.get('output', ''),
            'reasoning_depth': ex.get('metadata_reasoning_depth', 0),
            'split': ex.get('metadata_split', ''),
        }
        
        if dataset_name == 'proofwriter':
            record['num_facts'] = ex.get('metadata_num_facts', 0)
            record['num_rules'] = ex.get('metadata_num_rules', 0)
            record['proof_sample'] = ex.get('metadata_all_proofs', '')[:200 if not SHOW_FULL_PROOF else None]
        elif dataset_name == 'clutrr':
            record['f_comb'] = ex.get('metadata_f_comb', '')
            record['task_name'] = ex.get('metadata_task_name', '')
            record['proof_sample'] = ex.get('metadata_proof_state', '')[:200 if not SHOW_FULL_PROOF else None]
        
        all_examples.append(record)

logger.info(f"Total examples extracted: {len(all_examples)}")

## 7. Display Sample Examples
Show the first example from each dataset with its fields.

In [ ]:
for dataset_name in ['proofwriter', 'clutrr']:
    matching = [ex for ex in all_examples if ex['dataset'] == dataset_name]
    if matching:
        ex = matching[0]
        print(f"\n{'='*60}")
        print(f"Sample {dataset_name.upper()} Example")
        print(f"{'='*60}")
        print(f"Input:\n{ex['input'][:100 if not SHOW_FULL_INPUT else None]}..." if len(ex['input']) > 100 and not SHOW_FULL_INPUT else f"Input:\n{ex['input']}")
        print(f"\nOutput: {ex['output']}")
        print(f"Reasoning Depth: {ex['reasoning_depth']}")
        print(f"Split: {ex['split']}")
        if dataset_name == 'proofwriter':
            print(f"Num Facts: {ex['num_facts']}, Num Rules: {ex['num_rules']}")
            print(f"Proof Trace Sample: {ex['proof_sample']}")
        else:
            print(f"F_comb: {ex['f_comb']}, Task: {ex['task_name']}")
            print(f"Proof State Sample: {ex['proof_sample']}")

## 8. Statistical Summary
Create a summary table of dataset statistics.

In [ ]:
stats = []
for dataset_name in ['proofwriter', 'clutrr']:
    matching = [ex for ex in all_examples if ex['dataset'] == dataset_name]
    if matching:
        depths = [ex['reasoning_depth'] for ex in matching]
        stats.append({
            'Dataset': dataset_name,
            'Count': len(matching),
            'Avg Depth': f"{np.mean(depths):.1f}",
            'Max Depth': max(depths),
            'Min Depth': min(depths),
            'Splits': ', '.join(set(ex['split'] for ex in matching))
        })

stats_df = pd.DataFrame(stats)
print("\n" + "="*80)
print("Dataset Statistics")
print("="*80)
print(stats_df.to_string(index=False))

## 9. Visualization: Reasoning Depth Distribution
Plot the distribution of reasoning depths across datasets.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for idx, dataset_name in enumerate(['proofwriter', 'clutrr']):
    matching = [ex for ex in all_examples if ex['dataset'] == dataset_name]
    if matching:
        depths = [ex['reasoning_depth'] for ex in matching]
        axes[idx].hist(depths, bins=range(int(min(depths)), int(max(depths)) + 2), edgecolor='black', alpha=0.7, color='steelblue')
        axes[idx].set_title(f"{dataset_name.upper()} - Reasoning Depth Distribution")
        axes[idx].set_xlabel("Reasoning Depth")
        axes[idx].set_ylabel("Count")
        axes[idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

logger.info("Visualization complete")

## 10. Schema Validation & Summary
Verify data conforms to exp_sel_data_out schema and summarize findings.

In [ ]:
required_fields = {'input', 'output', 'metadata_reasoning_depth', 'metadata_split'}
validation_results = []

for ds_info in data['datasets']:
    dataset_name = ds_info['dataset']
    examples = ds_info['examples'][:MAX_EXAMPLES_PER_DATASET]
    
    valid_count = 0
    invalid_count = 0
    
    for ex in examples:
        if all(key in ex for key in required_fields):
            valid_count += 1
        else:
            invalid_count += 1
    
    validation_results.append({
        'Dataset': dataset_name,
        'Valid Records': valid_count,
        'Invalid Records': invalid_count,
        'Validation %': f"{100 * valid_count / (valid_count + invalid_count):.1f}%" if (valid_count + invalid_count) > 0 else "N/A"
    })

val_df = pd.DataFrame(validation_results)
print("\n" + "="*80)
print("Schema Validation Results")
print("="*80)
print(val_df.to_string(index=False))
print(f"\nAll {sum(r['Valid Records'] for r in validation_results)} records passed schema validation.")

## Summary

### What this notebook demonstrates:
1. **Data Loading**: Loading ProofWriter + CLUTRR datasets from a unified JSON schema
2. **Standardization**: Both datasets are mapped to `exp_sel_data_out` format with consistent fields
3. **Reasoning Traces**: Gold proof traces available for evaluating neuro-symbolic reasoning
4. **Reasoning Depth**: Both datasets span varying depths of multi-hop reasoning

### Key Statistics:
- **ProofWriter**: Propositional logic reasoning with gold proof traces (gold standard for trace auditing)
- **CLUTRR**: Kinship/relational reasoning with explicit proof chains (complementary reasoning modality)
- **Combined Total**: 325K+ examples across train/test splits

### To scale up:
Increase `MAX_EXAMPLES_PER_DATASET` in the Configuration cell to load more examples. The original full dataset has:
- ProofWriter: ~230K examples
- CLUTRR: ~95K examples

This demo runs on the first 3 examples from each dataset (~6 examples total) for quick iteration.